# 🎨 COCOKIN AI — Training Model Personal Color & Skin Tone
**Proyek IPPL & PCD (Pengolahan Citra Digital)**

Notebook ini melakukan pipeline lengkap Machine Learning & Citra Digital:
1. **Download Dataset Foto Wajah Asli** dari web (berdasarkan 4 musim Personal Color).
2. **Deteksi & Segmentasi Kulit Wajah (ROI Pipi/Dahi)** menggunakan OpenCV & MediaPipe (menghilangkan background & rambut).
3. **Ekstraksi Fitur Warna Kulit** (Color Spaces: RGB, HSV, dan CIE-L*a*b*).
4. **Training Model Klasifikasi** (Random Forest Classifier).
5. **Evaluasi Akurasi & Export Otak AI (`cocokin_ai_model.pkl`)** untuk backend aplikasi.

## Langkah 1: Install & Import Library Pendukung Citra Digital

In [ ]:
# Install pendeteksi wajah dan downloader gambar
!pip install -q opencv-python mediapipe scikit-learn pandas requests joblib

import os
import cv2
import requests
import numpy as np
import pandas as pd
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import matplotlib.pyplot as plt

print("✅ Library berhasil dimuat!")

## Langkah 2: Download Sampel Foto Wajah Asli dari Web
Mengunduh foto potret wajah nyata dengan variasi skin tone & undertone mewakili:
- **Warm Autumn** (Warm undertone, deep/medium warm)
- **Cool Summer** (Cool undertone, soft/pink base)
- **Warm Spring** (Warm cerah, peach/golden base)
- **Cool Winter** (Cool tajam, kontras tinggi)

In [ ]:
# Dataset foto wajah asli dari Unsplash (Royalty-free curated portraits)
IMAGE_DATASET = {
    "Warm Autumn": [
        "https://images.unsplash.com/photo-1534528741775-53994a69daeb?w=500&q=80",
        "https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=500&q=80",
        "https://images.unsplash.com/photo-1500648767791-00dcc994a43e?w=500&q=80",
        "https://images.unsplash.com/photo-1573496359142-b8d87734a5a2?w=500&q=80"
    ],
    "Cool Summer": [
        "https://images.unsplash.com/photo-1494790108377-be9c29b29330?w=500&q=80",
        "https://images.unsplash.com/photo-1438761681033-6461ffad8d80?w=500&q=80",
        "https://images.unsplash.com/photo-1544005313-94ddf0286df2?w=500&q=80",
        "https://images.unsplash.com/photo-1524504388940-b1c1722653e1?w=500&q=80"
    ],
    "Warm Spring": [
        "https://images.unsplash.com/photo-1517841905240-472988babdf9?w=500&q=80",
        "https://images.unsplash.com/photo-1539571696357-5a69c17a67c6?w=500&q=80",
        "https://images.unsplash.com/photo-1506794778202-cad84cf45f1d?w=500&q=80",
        "https://images.unsplash.com/photo-1529626455594-4ff0802cfb7e?w=500&q=80"
    ],
    "Cool Winter": [
        "https://images.unsplash.com/photo-1531746020798-e6953c6e8e04?w=500&q=80",
        "https://images.unsplash.com/photo-1509967419530-da38b4704bc6?w=500&q=80",
        "https://images.unsplash.com/photo-1531123897727-8f129e1688ce?w=500&q=80",
        "https://images.unsplash.com/photo-1508214751196-bcfd4ca60f91?w=500&q=80"
    ]
}

os.makedirs("dataset_wajah", exist_ok=True)
total_downloaded = 0

for label, urls in IMAGE_DATASET.items():
    category_dir = os.path.join("dataset_wajah", label.replace(" ", "_"))
    os.makedirs(category_dir, exist_ok=True)
    for idx, url in enumerate(urls):
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                file_path = os.path.join(category_dir, f"face_{idx+1}.jpg")
                with open(file_path, "wb") as f:
                    f.write(resp.content)
                total_downloaded += 1
        except Exception as e:
            print(f"Gagal unduh {url}: {e}")

print(f"✅ Sukses mengunduh {total_downloaded} foto wajah nyata ke direktori Colab!")

## Langkah 3: Ekstraksi Fitur Kulit Wajah (Pengolahan Citra Digital - PCD)
Menggunakan teknik deteksi wajah untuk memotong Region of Interest (ROI) pada bagian pipi dan dahi, lalu menghitung rata-rata nilai piksel:
- **RGB** (Red, Green, Blue)
- **HSV** (Hue = rona warna, Saturation = kepekatan, Value = kecerahan)
- **CIE-L*a*b*** (L = Luminance/Skin Tone, a* = Red-Green, b* = Yellow-Blue/Undertone)

In [ ]:
mp_face = mp.solutions.face_detection
detector = mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.5)

def extract_skin_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = detector.process(rgb)
    
    # Jika wajah terdeteksi, ambil area tengah wajah (pipi/hidung)
    h, w, _ = img.shape
    if results.detections:
        bbox = results.detections[0].location_data.relative_bounding_box
        x = int(bbox.xmin * w)
        y = int(bbox.ymin * h)
        bw = int(bbox.width * w)
        bh = int(bbox.height * h)
        
        # Ambil ROI pipi (area 30% di tengah wajah agar murni kulit)
        roi_x = max(0, x + int(bw * 0.35))
        roi_y = max(0, y + int(bh * 0.4))
        roi_w = int(bw * 0.3)
        roi_h = int(bh * 0.2)
        skin_roi = rgb[roi_y:roi_y+roi_h, roi_x:roi_x+roi_w]
    else:
        # Fallback jika wajah miring: ambil area crop tengah
        skin_roi = rgb[int(h*0.35):int(h*0.65), int(w*0.35):int(w*0.65)]
        
    if skin_roi.size == 0:
        return None
        
    # Konversi color space
    hsv_roi = cv2.cvtColor(skin_roi, cv2.COLOR_RGB2HSV)
    lab_roi = cv2.cvtColor(skin_roi, cv2.COLOR_RGB2LAB)
    
    # Hitung rata-rata piksel
    mean_r, mean_g, mean_b = np.mean(skin_roi, axis=(0, 1))
    mean_h, mean_s, mean_v = np.mean(hsv_roi, axis=(0, 1))
    mean_l, mean_a, mean_b_val = np.mean(lab_roi, axis=(0, 1))
    
    return [mean_r, mean_g, mean_b, mean_h, mean_s, mean_v, mean_l, mean_a, mean_b_val]

# Ekstraksi seluruh dataset foto
rows = []
for label, urls in IMAGE_DATASET.items():
    category_dir = os.path.join("dataset_wajah", label.replace(" ", "_"))
    for file_name in os.listdir(category_dir):
        p = os.path.join(category_dir, file_name)
        features = extract_skin_features(p)
        if features:
            rows.append(features + [label])

# Augmentasi data ringan (variasi pencahayaan alami) untuk memperkuat akurasi training
augmented_rows = []
for r in rows:
    feats = np.array(r[:-1])
    label = r[-1]
    augmented_rows.append(r)
    for _ in range(15): # Variasi delta brightness/shadow
        noise = np.random.normal(0, 2.5, len(feats))
        augmented_rows.append(list(np.clip(feats + noise, 0, 255)) + [label])

columns = ['R', 'G', 'B', 'H', 'S', 'V', 'L', 'a', 'b', 'Personal_Color']
df = pd.DataFrame(augmented_rows, columns=columns)
print(f"✅ Sukses mengekstrak fitur piksel kulit! Total sampel terproses: {len(df)}")
df.head()

## Langkah 4: Pelatihan Model Machine Learning (Training)

In [ ]:
X = df[['R', 'G', 'B', 'H', 'S', 'V', 'L', 'a', 'b']]
y = df['Personal_Color']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred) * 100

print(f"🎯 Akurasi Model: {acc:.2f}%")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred))

## Langkah 5: Uji Coba Inferensi & Export Otak AI (`.pkl`)
Menyimpan model yang sudah terlatih agar bisa di-download dan diintegrasikan ke backend Cocokin.

In [ ]:
# Fungsi uji coba prediksi
def predict_personal_color(r, g, b, h, s, v, l, a, b_val):
    input_data = np.array([[r, g, b, h, s, v, l, a, b_val]])
    hasil = model.predict(input_data)[0]
    prob = np.max(model.predict_proba(input_data)) * 100
    
    undertone = "Warm" if "Warm" in hasil else "Cool"
    skin_tone = "Fair" if l > 70 else ("Medium" if l > 55 else "Deep")
    
    return {
        "Personal_Color": hasil,
        "Confidence": f"{prob:.1f}%",
        "Undertone": undertone,
        "Skin_Tone": skin_tone
    }

# Contoh prediksi menggunakan salah satu data uji
sample = X_test.iloc[0]
print("Hasil Prediksi Sample Wajah:", predict_personal_color(*sample))

# Simpan file model .pkl
joblib.dump(model, "cocokin_ai_model.pkl")
print("\n🚀 Model berhasil disimpan ke file: cocokin_ai_model.pkl")
print("Silakan download file 'cocokin_ai_model.pkl' dari menu Files di sebelah kiri Google Colab!")